In [1]:
#acces a servidor
import os
import requests
#avg_no2 --> avg
#detailed_no2 --> hores
#SELECT nom_estacio, "data", contaminant, unitats, latitud, longitud, no2
#FROM "others".no2_april_2007;

option_db = '"others".no2_april_2007'
pollutant = requests.get('http://mappo-server.herokuapp.com/database?option='+option_db)


pollutant = str(pollutant.content)
primer_caracter = False
last_char = ''
#print(pollutant)
with open(option_db+'.csv', 'w') as f:

    if "detailed" not in option_db:
        f.write("estacio,data,contaminant,unitats,latitud,longitud,valor,\n")
    else:
        f.write("estacio,data,contaminant,unitats,latitud,longitud,quantity,\n")

    for c in pollutant:
        if c == '(':
            primer_caracter = True
            
        if (c >= '0' and c <= '9' or c >= 'A' and c <= 'Z' or c >= 'a' and c <= 'z' or c == '.' or c == '-' or c == ',' or c == ' ' or c == ':') and primer_caracter==True:
            last_char = c
            f.write(c)

        elif c == ')' and primer_caracter == True:
            f.write(',')
            last_char = c
            f.write('\n')
            primer_caracter = False

In [3]:
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from datetime import date
from sklearn.preprocessing import LabelEncoder
from itertools import islice

#Dataset
dataset = pd.read_csv('/content/"others".no2_april_2007.csv', sep = ',')#index_col=1
df = pd.DataFrame(dataset, columns=['estacio','data', 'contaminant', 'unitats', 'latitud', 'longitud', 'valor'])
df.fillna(method="ffill", inplace=True)
df = df.sort_values(by='data')

df.head(20)

,estacio,data,contaminant,unitats,latitud,longitud,valor


In [ ]:
#Dataset to arrays
valores = df['valor']
dates = df['data']
suma = np.zeros(len(valores), float)
dates.reset_index(drop=True, inplace=True)
valores.reset_index(drop=True, inplace=True)
fecha = []
print(dates)

#Computing daily avg 
multiple = 8
ultim = 943
for i in range(len(valores)):
  if i%8 == 0 and i!=0 or i==ultim:    
    if i==ultim:
      fecha.append(dates[i-multiple+1]) 
      suma[i-multiple] = (sum(valores[(i-multiple+1):(i+1)])/multiple)
    else:
      fecha.append(dates[i-multiple])
      suma[i-multiple] = (sum(valores[(i-multiple):i])/multiple)

avg = suma[suma!=0]
#print((avg))
#print(fecha)

dfd = pd.DataFrame({'fecha': fecha, 'avg': list(avg)}, index=fecha, columns=['avg'])
print(dfd.size)